# NB_MON_01 · Evaluar alertas de la plataforma

Junta en un solo lugar lo que hoy está repartido en cada caso y decide qué merece una alerta:

| Fuente | Dónde | Caso |
|---|---|---|
| Corridas y pasos | `mon.v_corridas`, `mon.v_pasos` (sobre `log.*`) | Pipeline medallion |
| SLA por grupo | `mon.SLA_CARGA` | Este caso |
| Frescura por entidad | `mon.v_frescura_entidad` (sobre `meta.ENTITY` + `log.RUN_STEP`) | Pipeline medallion |
| Inventario | `mon.v_inventario` (sobre `config.DOMAIN_OBJECTS`), si existe | Inventario |
| Evaluaciones del agente | `agente_ctl.eval_historial` en el lakehouse, si existe | Agente |
| Comparaciones de performance | `perf.comparacion` en el lakehouse, si existe | Performance |
| Reglas de calidad | `dq.v_ultimo_resultado`, si existe | Calidad y reconciliación |

Flujo: lee las fuentes → aplica las reglas (celda `# --- funciones`) → escribe la foto en `mon.STG_ALERTA` → `mon.usp_SYNC_ALERTAS` abre, actualiza y cierra alertas → deja el parte en `Files/monitoreo/partes/` para la skill `diagnosticar-plataforma`.

Escribe **solo** en el esquema `mon` de la base de control y en `Files/monitoreo/`. No toca `log`, `meta` ni `config`.
Requisito: `sql/01_ddl_monitoreo.sql` corrido en la base de control.

## Parámetros

In [ ]:
# Parámetros (celda marcada como "parameters": se pueden pasar desde un pipeline)
SQL_SERVER   = "<servidor>.database.fabric.microsoft.com"   # SQL database > Settings > Connection strings
SQL_DATABASE = "<nombre-de-la-base>"                         # DB_TDM_Platform

DIAS_HISTORIA = 30                    # pasos de silver que se leen para comparar volumen
TABLA_EVAL_AGENTE = "agente_ctl.eval_historial"
TABLA_PERF_COMPARACION = "perf.comparacion"
CARPETA_PARTES = "/lakehouse/default/Files/monitoreo/partes"
UMBRALES = {}                         # pisa los de UMBRALES_POR_DEFECTO, ej. {"max_pct_rechazo": 2.0}

DRY_RUN = False                       # True: evalúa y muestra, pero no escribe en mon ni en Files
FALLAR_SI_HAY_ALTA_NUEVA = True       # el notebook termina con error si aparece una alerta ALTA nueva:
                                      # así el pipeline falla y dispara su aviso

## Reglas

In [ ]:
# --- funciones (sin Fabric: las usa pruebas/prueba_local.py)
# Cada regla recibe DataFrames de pandas con las columnas de las vistas mon.v_* o de las
# tablas Delta de cada caso, y devuelve una lista de alertas (dict). Ninguna escribe nada.
import datetime as dt

import pandas as pd

SEVERIDADES = ("ALTA", "MEDIA", "BAJA")

UMBRALES_POR_DEFECTO = {
    "ventana_horas": 24,             # pasos fallidos e ítems borrados que se miran hacia atrás
    "max_duracion_min": 180,         # corrida RUNNING colgada, si el grupo no tiene MAX_DURACION_MIN
    "max_pct_rechazo": 5.0,          # % de filas rechazadas por calidad en silver
    "dias_sin_datos": 3,             # entidad activa sin traer filas a silver
    "volumen_min_ratio": 0.5,        # filas de la última carga contra la mediana de las anteriores
    "volumen_max_ratio": 3.0,
    "volumen_min_historia": 5,       # cargas anteriores necesarias para comparar volumen
    "dias_inventario": 2,            # inventario sin correr
    "dias_sin_evaluar_agente": 7,    # agente sin evaluación
}


def alerta(origen, severidad, codigo, objeto, mensaje, clave_extra=""):
    assert severidad in SEVERIDADES, severidad
    clave = "|".join(x for x in (codigo, str(objeto), clave_extra) if x)
    return {"CLAVE": clave[:300], "ORIGEN": origen, "SEVERIDAD": severidad, "CODIGO": codigo,
            "OBJETO": str(objeto)[:300], "MENSAJE": str(mensaje)[:2000]}


def _ts(serie):
    return pd.to_datetime(serie, errors="coerce", utc=True).dt.tz_localize(None)


def _vacio(df):
    return df is None or len(df) == 0


# ---------------------------------------------------------------- MDP
def alertas_mdp_fallas(pasos, ahora, u):
    """Último estado de cada (capa, objeto) dentro de la ventana: si es FAILED, alerta.
    Si después de fallar volvió a correr bien, no alerta."""
    if _vacio(pasos):
        return []
    p = pasos.copy()
    p["LOGGED_AT"] = _ts(p["LOGGED_AT"])
    p = p[p["LOGGED_AT"] >= ahora - dt.timedelta(hours=u["ventana_horas"])]
    if p.empty:
        return []
    ultimos = p.sort_values("LOGGED_AT").groupby(["LAYER", "OBJECT_CODE"], as_index=False).tail(1)
    out = []
    for _, r in ultimos[ultimos["STATUS"] == "FAILED"].iterrows():
        err = (r.get("ERROR_MESSAGE") or "sin mensaje")
        err = str(err).strip().splitlines()[0][:300] if str(err).strip() else "sin mensaje"
        out.append(alerta("MDP", "ALTA", "MDP_PASO_FALLIDO", f"{r['LAYER']}.{r['OBJECT_CODE']}",
                          f"Falló {r['LAYER']} de {r['OBJECT_CODE']} (grupo {r.get('LOAD_GROUP', '?')}, "
                          f"corrida {r['RUN_ID']}, {r['LOGGED_AT']:%Y-%m-%d %H:%M} UTC): {err}"))
    return out


def alertas_mdp_colgadas(corridas, sla, ahora, u):
    if _vacio(corridas):
        return []
    limites = {}
    if not _vacio(sla):
        limites = {r["LOAD_GROUP"]: r["MAX_DURACION_MIN"] for _, r in sla.iterrows()
                   if pd.notna(r.get("MAX_DURACION_MIN"))}
    c = corridas[corridas["STATUS"] == "RUNNING"].copy()
    c["STARTED_AT"] = _ts(c["STARTED_AT"])
    out = []
    for _, r in c.iterrows():
        maximo = int(limites.get(r["LOAD_GROUP"], u["max_duracion_min"]))
        minutos = (ahora - r["STARTED_AT"]).total_seconds() / 60
        if minutos > maximo:
            out.append(alerta("MDP", "ALTA", "MDP_CORRIDA_COLGADA", r["RUN_ID"],
                              f"La corrida {r['RUN_ID']} del grupo {r['LOAD_GROUP']} lleva {minutos:.0f} min "
                              f"en RUNNING (máximo {maximo}). Revisá el Monitoring hub: puede haber "
                              "terminado sin llamar a log.usp_END_RUN."))
    return out


def alertas_mdp_sla(corridas, sla, ahora):
    """Para cada grupo con SLA activo hoy y la hora límite ya pasada: ¿hubo una corrida
    SUCCEEDED que terminó hoy antes de la hora límite?"""
    if _vacio(sla):
        return []
    c = corridas.copy() if not _vacio(corridas) else pd.DataFrame(columns=["LOAD_GROUP", "STATUS", "ENDED_AT"])
    c["ENDED_AT"] = _ts(c["ENDED_AT"])
    hoy = ahora.date()
    out = []
    for _, s in sla.iterrows():
        if not bool(s.get("IS_ACTIVE", True)) or str(ahora.isoweekday()) not in str(s["DIAS_SEMANA"]):
            continue
        hh, mm = (int(x) for x in str(s["HORA_LIMITE_UTC"]).split(":"))
        limite = dt.datetime.combine(hoy, dt.time(hh, mm))
        if ahora < limite:
            continue
        ok = c[(c["LOAD_GROUP"] == s["LOAD_GROUP"]) & (c["STATUS"] == "SUCCEEDED")
               & (c["ENDED_AT"].dt.date == hoy)]
        if len(ok) and ok["ENDED_AT"].min() <= limite:
            continue
        dueno = f" Avisar a: {s['DUENO']}." if pd.notna(s.get("DUENO")) and s.get("DUENO") else ""
        if len(ok):
            out.append(alerta("MDP", "MEDIA", "MDP_SLA_TARDE", s["LOAD_GROUP"],
                              f"El grupo {s['LOAD_GROUP']} terminó bien a las {ok['ENDED_AT'].min():%H:%M} UTC, "
                              f"después de la hora límite {s['HORA_LIMITE_UTC']} UTC.{dueno}", hoy.isoformat()))
        else:
            out.append(alerta("MDP", "ALTA", "MDP_SLA_INCUMPLIDO", s["LOAD_GROUP"],
                              f"El grupo {s['LOAD_GROUP']} no tiene una corrida exitosa hoy y ya pasó la "
                              f"hora límite {s['HORA_LIMITE_UTC']} UTC.{dueno}", hoy.isoformat()))
    return out


def alertas_mdp_rechazos(pasos, ahora, u):
    if _vacio(pasos):
        return []
    p = pasos[pasos["LAYER"] == "silver"].copy()
    p["LOGGED_AT"] = _ts(p["LOGGED_AT"])
    p = p[(p["LOGGED_AT"] >= ahora - dt.timedelta(hours=u["ventana_horas"])) & (p["STATUS"] == "SUCCEEDED")]
    if p.empty:
        return []
    ultimos = p.sort_values("LOGGED_AT").groupby("OBJECT_CODE", as_index=False).tail(1)
    out = []
    for _, r in ultimos.iterrows():
        leidas, rech = pd.to_numeric(r["ROWS_READ"], errors="coerce"), pd.to_numeric(r["ROWS_REJECTED"], errors="coerce")
        if pd.isna(leidas) or leidas <= 0 or pd.isna(rech):
            continue
        pct = 100 * rech / leidas
        if pct > u["max_pct_rechazo"]:
            out.append(alerta("MDP", "MEDIA", "MDP_RECHAZOS_ALTOS", r["OBJECT_CODE"],
                              f"{r['OBJECT_CODE']}: {int(rech)} de {int(leidas)} filas rechazadas por calidad "
                              f"({pct:.1f} %, umbral {u['max_pct_rechazo']} %). Ver silver.dq_rechazos."))
    return out


def alertas_mdp_volumen(pasos, u):
    """Última carga exitosa de silver contra la mediana de las anteriores (solo con historia suficiente)."""
    if _vacio(pasos):
        return []
    p = pasos[(pasos["LAYER"] == "silver") & (pasos["STATUS"] == "SUCCEEDED")].copy()
    p["LOGGED_AT"] = _ts(p["LOGGED_AT"])
    p["ROWS_WRITTEN"] = pd.to_numeric(p["ROWS_WRITTEN"], errors="coerce")
    p = p.dropna(subset=["ROWS_WRITTEN"])
    out = []
    for obj, g in p.sort_values("LOGGED_AT").groupby("OBJECT_CODE"):
        if len(g) < u["volumen_min_historia"] + 1:
            continue
        ultima, mediana = g["ROWS_WRITTEN"].iloc[-1], g["ROWS_WRITTEN"].iloc[:-1].median()
        if mediana <= 0:
            continue
        ratio = ultima / mediana
        if ratio < u["volumen_min_ratio"] or ratio > u["volumen_max_ratio"]:
            out.append(alerta("MDP", "BAJA", "MDP_VOLUMEN_ANOMALO", obj,
                              f"{obj}: la última carga escribió {int(ultima)} filas contra una mediana de "
                              f"{mediana:.0f} en las {len(g) - 1} anteriores (x{ratio:.2f})."))
    return out


def alertas_mdp_frescura(frescura, ahora, u):
    if _vacio(frescura):
        return []
    f = frescura.copy()
    f["ULTIMA_CARGA_CON_DATOS"] = _ts(f["ULTIMA_CARGA_CON_DATOS"])
    limite = ahora - dt.timedelta(days=u["dias_sin_datos"])
    out = []
    for _, r in f.iterrows():
        ult = r["ULTIMA_CARGA_CON_DATOS"]
        if pd.isna(ult):
            msg = f"{r['ENTITY_CODE']} está activa y nunca trajo filas a silver."
        elif ult < limite:
            msg = (f"{r['ENTITY_CODE']} no trae filas nuevas a silver desde {ult:%Y-%m-%d %H:%M} UTC "
                   f"(más de {u['dias_sin_datos']} días). Puede ser normal en un origen que cambia poco.")
        else:
            continue
        out.append(alerta("MDP", "MEDIA", "MDP_ENTIDAD_SIN_DATOS", r["ENTITY_CODE"], msg))
    return out


# ---------------------------------------------------------------- Inventario
def alertas_inventario(inv, ahora, u):
    if _vacio(inv):
        return []
    i = inv.copy()
    for c in ("DELETED_AT", "LAST_SEEN_AT"):
        i[c] = _ts(i[c])
    out = []
    ultima = i["LAST_SEEN_AT"].max()
    if pd.isna(ultima) or ultima < ahora - dt.timedelta(days=u["dias_inventario"]):
        cuando = "nunca" if pd.isna(ultima) else f"desde {ultima:%Y-%m-%d}"
        out.append(alerta("INV", "BAJA", "INV_SIN_CORRER", "config.DOMAIN_OBJECTS",
                          f"NB_INV_01 no actualiza el inventario {cuando}."))
    borrados = i[i["DELETED_AT"] >= ahora - dt.timedelta(hours=u["ventana_horas"])]
    for ws, g in borrados.groupby("WORKSPACE_ID"):
        nombres = ", ".join(f"{n} ({t})" for n, t in zip(g["OBJECT_NAME"], g["ARTIFACT_TYPE"]))
        out.append(alerta("INV", "MEDIA", "INV_ITEMS_BORRADOS", ws,
                          f"{len(g)} items dejaron de existir en el workspace {ws}: {nombres[:1500]}"))
    return out


# ---------------------------------------------------------------- Agente
def alertas_agente(ev, ahora, u):
    if _vacio(ev):
        return []
    e = ev.copy()
    e["evaluado_en"] = _ts(e["evaluado_en"])
    out = []
    for agente, g in e.sort_values("evaluado_en").groupby("agente"):
        r = g.iloc[-1]
        if not bool(r["aprueba"]):
            sev = "ALTA" if r["etapa"] == "production" else "MEDIA"
            out.append(alerta("AGT", sev, "AGT_NO_APRUEBA", agente,
                              f"{agente} ({r['etapa']}) no aprueba la evaluación del {r['evaluado_en']:%Y-%m-%d}: "
                              f"acierto {float(r['acierto']):.1%}. {r.get('motivos') or ''}".strip()))
        if r["evaluado_en"] < ahora - dt.timedelta(days=u["dias_sin_evaluar_agente"]):
            out.append(alerta("AGT", "BAJA", "AGT_SIN_EVALUAR", agente,
                              f"{agente} no se evalúa desde {r['evaluado_en']:%Y-%m-%d}."))
    return out


# ---------------------------------------------------------------- Performance
def alertas_perf(comp):
    """perf.comparacion guarda todo como texto. Se mira la última comparación de cada modelo."""
    if _vacio(comp):
        return []
    c = comp.copy()
    c["fecha_utc"] = _ts(c["fecha_utc"])
    out = []
    for modelo, g in c.groupby("modelo"):
        ultima = g[g["fecha_utc"] == g["fecha_utc"].max()]
        malas = ultima[ultima["estado"].isin(["regresion", "resultado_distinto"])]
        if len(malas):
            r = ultima.iloc[0]
            ids = ", ".join(f"{i} ({e})" for i, e in zip(malas["id"], malas["estado"]))
            out.append(alerta("PERF", "MEDIA", "PERF_REGRESION", modelo,
                              f"{modelo}: {r['corrida_antes']} → {r['corrida_despues']} "
                              f"(veredicto {r['veredicto']}) con consultas que empeoran o cambian de resultado: "
                              f"{ids[:1500]}"))
    return out


# ---------------------------------------------------------------- Calidad
def alertas_calidad(ultimos):
    """dq.v_ultimo_resultado: el último resultado de cada regla activa (Calidad_Reconciliacion)."""
    if _vacio(ultimos):
        return []
    out = []
    for _, r in ultimos[ultimos["ESTADO"] != "OK"].iterrows():
        sev = "ALTA" if r["SEVERIDAD"] == "ERROR" and r["ESTADO"] == "FALLA" else "MEDIA"
        que = "no se pudo ejecutar" if r["ESTADO"] == "ERROR_EJECUCION" else "falla"
        dueno = f" Avisar a: {r['DUENO']}." if pd.notna(r.get("DUENO")) and r.get("DUENO") else ""
        out.append(alerta("CAL", sev, "CAL_REGLA_" + ("ERROR_EJECUCION" if r["ESTADO"] == "ERROR_EJECUCION" else "FALLA"),
                          r["REGLA_CODE"],
                          f"{r['REGLA_CODE']} ({r['CATEGORIA']}, {r['SEVERIDAD']}) {que}: {r.get('DETALLE') or ''}"
                          f"{dueno}".strip()))
    return out


# ---------------------------------------------------------------- todo junto
def evaluar(fuentes: dict, ahora: dt.datetime, umbrales: dict | None = None):
    """fuentes: {'corridas','pasos','sla','frescura','inventario','eval_agente','perf_comparacion','calidad'};
    la que falta (None) no se evalúa. Devuelve (alertas, origenes_evaluados)."""
    u = {**UMBRALES_POR_DEFECTO, **(umbrales or {})}
    f = {k: fuentes.get(k) for k in ("corridas", "pasos", "sla", "frescura", "inventario",
                                       "eval_agente", "perf_comparacion", "calidad")}
    alertas, origenes = [], []
    if f["corridas"] is not None and f["pasos"] is not None:
        origenes.append("MDP")
        alertas += alertas_mdp_fallas(f["pasos"], ahora, u)
        alertas += alertas_mdp_colgadas(f["corridas"], f["sla"], ahora, u)
        alertas += alertas_mdp_sla(f["corridas"], f["sla"], ahora)
        alertas += alertas_mdp_rechazos(f["pasos"], ahora, u)
        alertas += alertas_mdp_volumen(f["pasos"], u)
        alertas += alertas_mdp_frescura(f["frescura"], ahora, u)
    if f["inventario"] is not None:
        origenes.append("INV")
        alertas += alertas_inventario(f["inventario"], ahora, u)
    if f["eval_agente"] is not None:
        origenes.append("AGT")
        alertas += alertas_agente(f["eval_agente"], ahora, u)
    if f["perf_comparacion"] is not None:
        origenes.append("PERF")
        alertas += alertas_perf(f["perf_comparacion"])
    if f["calidad"] is not None:
        origenes.append("CAL")
        alertas += alertas_calidad(f["calidad"])
    orden = {s: i for i, s in enumerate(SEVERIDADES)}
    alertas.sort(key=lambda a: (orden[a["SEVERIDAD"]], a["ORIGEN"], a["CLAVE"]))
    vistas = set()
    unicas = [a for a in alertas if not (a["CLAVE"] in vistas or vistas.add(a["CLAVE"]))]
    return unicas, origenes


def parte(alertas, origenes, ahora, resumen_corridas=None):
    """Lo que lee la skill diagnosticar-plataforma: alertas y un resumen de las corridas."""
    cuenta = {s: sum(a["SEVERIDAD"] == s for a in alertas) for s in SEVERIDADES}
    return {"generado_utc": ahora.isoformat(timespec="seconds"), "origenes_evaluados": origenes,
            "alertas_por_severidad": cuenta, "alertas": alertas,
            "corridas_24h": resumen_corridas or []}

## 1. Leer la base de control

In [ ]:
import com.microsoft.sqlserver.jdbc.spark  # registra el conector de Fabric (mismo patrón que NB_INV_01)

url = (
    f"jdbc:sqlserver://{SQL_SERVER}:1433;"
    f"database={SQL_DATABASE};"
    "encrypt=true;"
    "trustServerCertificate=false;"
    "hostNameInCertificate=*.database.fabric.microsoft.com;"
    "loginTimeout=30;"
)


def leer(consulta):
    return (spark.read.format("com.microsoft.sqlserver.jdbc.spark")
            .option("url", url).option("dbtable", f"({consulta}) q").load().toPandas())


existe_inv = leer("SELECT OBJECT_ID('mon.v_inventario') AS id")["id"].notna().iloc[0]
existe_cal = leer("SELECT OBJECT_ID('dq.v_ultimo_resultado') AS id")["id"].notna().iloc[0]

fuentes = {
    "corridas": leer(f"SELECT * FROM mon.v_corridas WHERE STARTED_AT >= DATEADD(DAY, -{int(DIAS_HISTORIA)}, SYSUTCDATETIME())"),
    "pasos": leer(f"SELECT * FROM mon.v_pasos WHERE LOGGED_AT >= DATEADD(DAY, -{int(DIAS_HISTORIA)}, SYSUTCDATETIME())"),
    "sla": leer("SELECT * FROM mon.SLA_CARGA WHERE IS_ACTIVE = 1"),
    "frescura": leer("SELECT * FROM mon.v_frescura_entidad"),
    "inventario": leer("SELECT * FROM mon.v_inventario") if existe_inv else None,
    "calidad": leer("SELECT * FROM dq.v_ultimo_resultado") if existe_cal else None,
}
for k, v in fuentes.items():
    print(f"{k}: {'no existe' if v is None else len(v)}")

## 2. Leer las tablas del lakehouse (si existen)

In [ ]:
def leer_delta(tabla, columnas):
    if not spark.catalog.tableExists(tabla):
        print(f"{tabla}: no existe, no se evalúa")
        return None
    return spark.table(tabla).select(*columnas).toPandas()


fuentes["eval_agente"] = leer_delta(TABLA_EVAL_AGENTE, ["agente", "etapa", "evaluado_en", "acierto", "aprueba", "motivos"])
fuentes["perf_comparacion"] = leer_delta(TABLA_PERF_COMPARACION, ["modelo", "id", "estado", "corrida_antes",
                                                                  "corrida_despues", "veredicto", "fecha_utc"])

## 3. Evaluar

In [ ]:
ahora = dt.datetime.now(dt.timezone.utc).replace(tzinfo=None, microsecond=0)
alertas, origenes = evaluar(fuentes, ahora, UMBRALES)
print(f"Orígenes evaluados: {origenes} · {len(alertas)} alertas")
display(pd.DataFrame(alertas, columns=["SEVERIDAD", "ORIGEN", "CODIGO", "OBJETO", "MENSAJE", "CLAVE"]))

## 4. Sincronizar con `mon.ALERTA`

Staging y stored procedure por JDBC con un token de Entra ID de la identidad que corre el notebook (mismo patrón que `NB_INV_01`). Son pocas filas: se insertan con un `PreparedStatement`.

In [ ]:
resultado = {"NUEVAS": None, "NUEVAS_ALTA": None, "ABIERTAS": None, "RESUELTAS": None}
if DRY_RUN:
    print("DRY_RUN: no se escribe en mon.")
else:
    # [verificar] audiencia del token para SQL database en Fabric; si falla, probá "https://database.windows.net/.default"
    token = notebookutils.credentials.getToken("https://database.windows.net/")
    jvm = spark._jvm
    props = jvm.java.util.Properties()
    props.setProperty("accessToken", token)
    conn = jvm.java.sql.DriverManager.getConnection(url, props)
    try:
        conn.setAutoCommit(False)
        conn.createStatement().executeUpdate("DELETE FROM mon.STG_ALERTA")
        ps = conn.prepareStatement("INSERT mon.STG_ALERTA (CLAVE, ORIGEN, SEVERIDAD, CODIGO, OBJETO, MENSAJE) "
                                   "VALUES (?, ?, ?, ?, ?, ?)")
        for a in alertas:
            for i, c in enumerate(["CLAVE", "ORIGEN", "SEVERIDAD", "CODIGO", "OBJETO", "MENSAJE"], start=1):
                ps.setString(i, a[c])
            ps.addBatch()
        if alertas:
            ps.executeBatch()
        conn.commit()
        conn.setAutoCommit(True)
        cs = conn.prepareCall("{call mon.usp_SYNC_ALERTAS(?)}")
        cs.setString(1, ",".join(origenes))
        if cs.execute():
            rs = cs.getResultSet()
            while rs.next():
                resultado = {k: rs.getInt(k) for k in resultado}
    finally:
        conn.close()
    print(resultado)

## 5. Parte para la skill y salida

In [ ]:
import json
import os

ult = fuentes["corridas"].copy()
ult["STARTED_AT"] = _ts(ult["STARTED_AT"])
ult = ult[ult["STARTED_AT"] >= ahora - dt.timedelta(hours=24)]
columnas = ["RUN_ID", "LOAD_GROUP", "STATUS", "STARTED_AT", "ENDED_AT", "DURACION_MIN",
            "PASOS", "PASOS_FALLIDOS", "FILAS_ESCRITAS", "FILAS_RECHAZADAS"]
resumen_corridas = json.loads(ult[columnas].to_json(orient="records", date_format="iso"))

p = parte(alertas, origenes, ahora, resumen_corridas)
p["sincronizacion"] = resultado
if not DRY_RUN:
    os.makedirs(CARPETA_PARTES, exist_ok=True)
    ruta = f"{CARPETA_PARTES}/{ahora:%Y%m%d_%H%M}.json"
    with open(ruta, "w", encoding="utf-8") as f:
        json.dump(p, f, ensure_ascii=False, indent=2, default=str)
    print("Parte en", ruta)

salida = {"alertas": len(alertas), **p["alertas_por_severidad"], **{k.lower(): v for k, v in resultado.items()}}
if FALLAR_SI_HAY_ALTA_NUEVA and (resultado["NUEVAS_ALTA"] or 0) > 0:
    altas = [a["MENSAJE"] for a in alertas if a["SEVERIDAD"] == "ALTA"]
    raise RuntimeError(f"{resultado['NUEVAS_ALTA']} alertas ALTA nuevas: " + " | ".join(altas)[:3000])
notebookutils.notebook.exit(json.dumps(salida))